# EMPIRE Voice Clone — Piper trainer (free Colab GPU)

Trains a **Piper** voice that sounds like you, then exports a local `your-voice.onnx` you can drop into OpenHuman.

**Before you start:** Runtime → Change runtime type → **GPU (T4)**.

Steps: upload your recording → auto transcribe + segment → fine-tune from the Lessac base voice → download `empire-roland.onnx` + `.json`.

Total time on a T4: ~30–90 min depending on how much audio + training steps.

## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install dependencies

In [ ]:
!apt-get -qq install -y espeak-ng ffmpeg >/dev/null
!pip -q install faster-whisper soundfile librosa onnxruntime > /dev/null
!pip -q install piper-tts >/dev/null
# piper training pipeline
!git clone -q https://github.com/rhasspy/piper /content/piper || true
%cd /content/piper/src/python
!pip -q install -e . >/dev/null 2>&1 || true
!pip -q install 'numpy<2' 'torch' 'torchaudio' 'pytorch-lightning==1.9.5' 'onnx' >/dev/null
print('deps installed')

## 3 · Upload your recording
Run this, then pick your `roland-voice.m4a/.wav/.mp3` from your Mac.

In [ ]:
from google.colab import files
import os, glob
os.makedirs('/content/raw', exist_ok=True)
up = files.upload()
src = list(up.keys())[0]
!ffmpeg -y -i "$src" -ac 1 -ar 22050 /content/raw/voice.wav -loglevel error
import soundfile as sf
info = sf.info('/content/raw/voice.wav')
print('duration (min):', round(info.duration/60,1))

## 4 · Auto-segment + transcribe (Whisper)
Splits your long recording into short clips and writes a Piper-format transcript.

In [ ]:
from faster_whisper import WhisperModel
import soundfile as sf, os
os.makedirs('/content/dataset/wav', exist_ok=True)
model = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments, _ = model.transcribe('/content/raw/voice.wav', word_timestamps=False, vad_filter=True)
audio, sr = sf.read('/content/raw/voice.wav')
lines=[]; n=0
for seg in segments:
    text=seg.text.strip()
    if len(text) < 4: continue
    a=int(seg.start*sr); b=int(seg.end*sr)
    if b-a < sr*0.6 or b-a > sr*15: continue
    clip=audio[a:b]; name=f'{n:05d}.wav'
    sf.write(f'/content/dataset/wav/{name}', clip, sr)
    lines.append(f'{name}|{text}')
    n+=1
open('/content/dataset/metadata.csv','w').write('\n'.join(lines))
print('clips:', n)

## 5 · Preprocess for Piper + download base checkpoint
Fine-tunes from the US Lessac medium voice (same family OpenHuman already uses).

In [ ]:
%cd /content/piper/src/python
# base checkpoint (Lessac medium) to fine-tune from
!wget -q -O /content/lessac.ckpt https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt
!python -m piper_train.preprocess \
  --language en-us \
  --input-dir /content/dataset \
  --output-dir /content/train \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050
print('preprocess done')

## 6 · Fine-tune (the actual training)
Adjust `max_epochs` higher for better quality (more time). 1000–2000 is a good range for fine-tuning.

In [ ]:
!python -m piper_train \
  --dataset-dir /content/train \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --validation-split 0.0 --num-test-examples 0 \
  --max_epochs 1500 \
  --resume_from_checkpoint /content/lessac.ckpt \
  --checkpoint-epochs 200 \
  --precision 16

## 7 · Export to .onnx + download

In [ ]:
import glob, os
ckpt = sorted(glob.glob('/content/train/lightning_logs/version_*/checkpoints/*.ckpt'))[-1]
print('using', ckpt)
!python -m piper_train.export_onnx "$ckpt" /content/empire-roland.onnx
!cp /content/train/config.json /content/empire-roland.onnx.json
from google.colab import files
files.download('/content/empire-roland.onnx')
files.download('/content/empire-roland.onnx.json')
print('Done! Send empire-roland.onnx + .json back to Claude to install into OpenHuman.')